# 6 - Spatial strategies: 5-seed robustness

Rigorous follow-up to notebook 5. Trains each spatial strategy at **5 seeds** and reports the
**seed-averaged** fake-cloud MAE/RMSE and error-over-time, so we can tell whether the single-seed
ranking (random best) survives run-to-run training noise. Same region, channels, split, and shared
stats as nb5. Maps stay single-model in nb5 (averages have no per-pixel map). patch80 is excluded
(80x112 fails cuDNN on this GPU).

In [ ]:
import os, pickle
import numpy as np, xarray as xr, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import mindthegap as mtg

RECHUNKED = "/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr"
ORIGINAL  = os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr")

SUBSET = dict(lat=slice(25, 5), lon=slice(50, 78))     # ocean-heavier 80x112 (27% land), clean tiling
features = []
train_year, train_range, val_range, test_range = 2015, 3, 1, 1
TEST_YEAR = train_year + train_range + val_range        # 2019
DAY_BATCH = 100
SEEDS = [0, 1, 2, 3, 4]

MODEL_DIR = "models/spatial_strat"
os.makedirs(MODEL_DIR, exist_ok=True)

STRATS = {
    "nonoverlap":   dict(tile=(40, 56), overlap=(0, 0),   batch=16, sampling="grid"),
    "overlap":      dict(tile=(40, 56), overlap=(20, 28), batch=16, sampling="grid"),
    "random":       dict(tile=(40, 56), n_per_day=16, batch=16, sampling="random", min_ocean=0.0),
    "ocean_random": dict(tile=(40, 56), n_per_day=16, batch=16, sampling="random", min_ocean=0.5),
    "coast":        dict(tile=(40, 56), n_per_day=16, batch=16, sampling="coast", coast_frac=0.5, min_ocean=0.5),
    "coast_weighted": dict(tile=(40, 56), n_per_day=16, batch=16, sampling="coast_weighted", scale=8.0, floor=0.15, min_ocean=0.25),
}

def open_region(path):
    ds = xr.open_zarr(path, chunks={})
    if SUBSET is not None:
        ds = ds.sel(**SUBSET)
    ds = mtg.crop_to_multiple(ds, multiple=8)
    return ds.sel(time=slice(f"{train_year}-01-01",
                             f"{train_year+train_range+val_range+test_range}-01-01"))

In [ ]:
# shared stats (deterministic; identical to nb5's since same source/region/params)
_, SHARED_STATS = mtg.build_standardized_lazy(open_region(ORIGINAL), features, train_year, train_range,
                                              standardize_chl=True)
y_mean, y_std = SHARED_STATS["CHL"][0], SHARED_STATS["CHL"][1]
print("CHL mean/std:", y_mean, y_std)

In [ ]:
# test-year inputs (for the seed-averaged table) + full-record inputs (for the error-over-time plot)
orig_test = open_region(ORIGINAL).sel(time=str(TEST_YEAR))
ds_std_test, _ = mtg.build_standardized_lazy(orig_test, features, train_year, train_range,
                                             standardize_chl=True, stats=SHARED_STATS)
ds_std_test = ds_std_test.load()
_t = ds_std_test.time.values
DATES = pd.to_datetime(_t[np.linspace(0, len(_t) - 1, 10).astype(int)])

ds_std_full, _ = mtg.build_standardized_lazy(open_region(ORIGINAL), features, train_year, train_range,
                                             standardize_chl=True, stats=SHARED_STATS)   # lazy
TRAIN_END = f"{train_year + train_range}-01-01"                 # 2018-01-01 (train = 2015-2017)
VAL_END   = f"{train_year + train_range + val_range}-01-01"     # 2019-01-01
print("eval dates:", [str(d.date()) for d in DATES])

## Coast strategy: preview the possible boxes

Before the sweep, sanity-check the `coast` sampler: each rectangle is a candidate crop anchored so one edge
sits on the shoreline (red = shore on the right edge, orange = left, magenta = top), extending into the
ocean. This is the discrete pool `coast` samples from each day.

In [ ]:
import numpy as np, xarray as xr, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import ndimage

reg = mtg.crop_to_multiple(xr.open_zarr(ORIGINAL, chunks={}).sel(**SUBSET), multiple=8)
chlvar = "CHL" if "CHL" in reg else [v for v in reg.data_vars if "CHL" in v.upper()][0]
ocean = ~reg[chlvar].sel(time="2019").isnull().all("time").values
coast = ocean & ndimage.binary_dilation(~ocean)
H, W = ocean.shape
th, tw, N, min_ocean = 40, 56, 40, 0.5

def coast_crops(ocean, coast, th, tw, n, seed=0, min_ocean=0.5):
    rng = np.random.default_rng(seed)
    ys, xs = np.where(coast); H, W = ocean.shape; out = []
    for i in rng.choice(len(ys), size=n*6, replace=True):
        r, c = int(ys[i]), int(xs[i])
        cands = [(r-th//2, c-tw+1, "R"), (r-th//2, c, "L"), (r, c-tw//2, "T")]
        best, bf = None, -1
        for yy, xx, e in cands:
            yy = int(np.clip(yy, 0, H-th)); xx = int(np.clip(xx, 0, W-tw))
            f = ocean[yy:yy+th, xx:xx+tw].mean()
            if f > bf: bf, best = f, (yy, xx, e)
        if bf >= min_ocean: out.append(best)
        if len(out) >= n: break
    return out

crops = coast_crops(ocean, coast, th, tw, N, min_ocean=min_ocean)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(ocean, origin="upper", cmap="Blues", vmin=0, vmax=1.4)
ax.contour(coast.astype(float), levels=[0.5], colors="k", linewidths=0.6)
ecolor = {"R": "red", "L": "orange", "T": "magenta"}
for yy, xx, e in crops:
    ax.add_patch(Rectangle((xx-0.5, yy-0.5), tw, th, fill=False, edgecolor=ecolor[e], lw=1.3))
ax.set_title(f"{len(crops)} coast-anchored {th}x{tw} chunks (red=shore right, orange=left, magenta=top; black=shoreline)")
ax.set_xlabel("lon index"); ax.set_ylabel("lat index"); plt.tight_layout(); plt.show()
print(f"region {H}x{W}, ocean {ocean.mean():.0%}, coast pixels {int(coast.sum())}")

In [ ]:
%%writefile train_seed.py
"""Train one spatial-chunk strategy at one seed, own process (fresh GPU).
Masked loss (land/gap pixels excluded). sampling in {grid, random, coast, coast_weighted}.
Usage: python train_seed.py --strategy <name> --seed <int>"""
import os, argparse, pickle
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, xarray as xr, tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from scipy import ndimage
import mindthegap as mtg

def masked_mse(yt, yp):
    y = yt[..., 0:1]; m = yt[..., 1:2]
    return tf.reduce_sum(tf.square(y - yp) * m) / (tf.reduce_sum(m) + 1e-6)

def open_region(path, c):
    ds = xr.open_zarr(path, chunks={}).sel(**c["SUBSET"])
    ds = mtg.crop_to_multiple(ds, multiple=8)
    ty, tr, vr, te = c["train_year"], c["train_range"], c["val_range"], c["test_range"]
    return ds.sel(time=slice(f"{ty}-01-01", f"{ty+tr+vr+te}-01-01"))

def grid_positions(LAT, LON, th, tw, overlap):
    oh, ow = overlap; sh, sw = th - oh, tw - ow
    n_h = (LAT - th) // sh + 1; n_w = (LON - tw) // sw + 1
    return [(i * sh, j * sw) for i in range(n_h) for j in range(n_w)]

def random_positions(LAT, LON, th, tw, n, rng, ocean, min_ocean):
    out = []; tries = 0
    while len(out) < n and tries < n * 50:
        tries += 1
        yy = int(rng.integers(0, LAT - th + 1)); xx = int(rng.integers(0, LON - tw + 1))
        if min_ocean <= 0 or ocean[yy:yy+th, xx:xx+tw].mean() >= min_ocean:
            out.append((yy, xx))
    return out

def coast_positions(ocean, coast, th, tw, n, rng, min_ocean):
    ys, xs = np.where(coast); H, W = ocean.shape; out = []
    for i in rng.choice(len(ys), size=n * 8, replace=True):
        r, cc = int(ys[i]), int(xs[i]); best = None; bf = -1
        for yy, xx in [(r - th // 2, cc - tw + 1), (r - th // 2, cc), (r, cc - tw // 2)]:
            yy = int(np.clip(yy, 0, H - th)); xx = int(np.clip(xx, 0, W - tw))
            f = ocean[yy:yy+th, xx:xx+tw].mean()
            if f > bf: bf, best = f, (yy, xx)
        if bf >= min_ocean: out.append(best)
        if len(out) >= n: break
    return out

ap = argparse.ArgumentParser()
ap.add_argument("--strategy", required=True)
ap.add_argument("--seed", type=int, required=True)
ap.add_argument("--config", default="models/spatial_strat/strat_config.pkl")
a = ap.parse_args()
c = pickle.load(open(a.config, "rb")); S = c["STRATS"][a.strategy]
for g in tf.config.list_physical_devices("GPU"): tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(a.seed)

ty, tr, vr = c["train_year"], c["train_range"], c["val_range"]
ds = open_region(c["ORIGINAL"], c)
LAT, LON = ds.sizes["lat"], ds.sizes["lon"]
ds_std, _ = mtg.build_standardized_lazy(ds, c["features"], ty, tr, standardize_chl=True,
    stats=c["SHARED_STATS"], output_chunks={"time": c["DAY_BATCH"], "lat": LAT, "lon": LON})
x_vars = [v for v in ds_std.data_vars if v != "CHL"]; NC = len(x_vars)

dtr = ds_std.sel(time=slice(f"{ty}-01-01", f"{ty+tr}-01-01")).load()
dva = ds_std.sel(time=slice(f"{ty+tr}-01-01", f"{ty+tr+vr}-01-01")).load()

def to_XY(dd):
    X = np.stack([np.nan_to_num(dd[v].values, nan=0.0) for v in x_vars], -1).astype("float32")
    chl = dd["CHL"].values
    Y = np.stack([np.nan_to_num(chl, nan=0.0), np.isfinite(chl).astype("float32")], -1).astype("float32")
    return X, Y

Xtr, Ytr = to_XY(dtr); Xva, Yva = to_XY(dva)
ocean = ~np.isnan(dtr["CHL"].values).all(0)
coast = ocean & ndimage.binary_dilation(~ocean)
th, tw = S["tile"]; batch = S["batch"]

# coast-weighted sampling: sample crop centres by distance-to-coast (coast oversampled, floor keeps
# open ocean covered) -> flips random's centre bias toward the shore while still seeing everything.
CW_FLAT = None
if S["sampling"] == "coast_weighted":
    dist = ndimage.distance_transform_edt(~coast)
    ww = (np.exp(-dist / S.get("scale", 8.0)) + S.get("floor", 0.15)) * ocean
    CW_FLAT = (ww / ww.sum()).ravel()

def positions_for(rng):
    if S["sampling"] == "grid":
        return grid_positions(LAT, LON, th, tw, S.get("overlap") or (0, 0))
    if S["sampling"] == "coast":
        n = S["n_per_day"]; nc = int(round(n * S.get("coast_frac", 1.0)))
        return (coast_positions(ocean, coast, th, tw, nc, rng, S.get("min_ocean", 0.5))
                + random_positions(LAT, LON, th, tw, n - nc, rng, ocean, S.get("min_ocean", 0.5)))
    if S["sampling"] == "coast_weighted":
        out = []; mo = S.get("min_ocean", 0.25)
        for idx in rng.choice(CW_FLAT.size, size=S["n_per_day"] * 4, p=CW_FLAT):
            cy, cx = divmod(int(idx), LON)
            yy = int(np.clip(cy - th // 2, 0, LAT - th)); xx = int(np.clip(cx - tw // 2, 0, LON - tw))
            if ocean[yy:yy+th, xx:xx+tw].mean() >= mo: out.append((yy, xx))
            if len(out) >= S["n_per_day"]: break
        return out
    return random_positions(LAT, LON, th, tw, S["n_per_day"], rng, ocean, S.get("min_ocean", 0.0))

ppd = len(grid_positions(LAT, LON, th, tw, S.get("overlap") or (0, 0))) if S["sampling"] == "grid" else S["n_per_day"]

def make_gen(Xf, Yf, seed):
    rng = np.random.default_rng(seed)
    def g():
        for d in range(Xf.shape[0]):
            for yy, xx in positions_for(rng):
                yield Xf[d, yy:yy+th, xx:xx+tw], Yf[d, yy:yy+th, xx:xx+tw]
    return g

sig = (tf.TensorSpec((th, tw, NC), tf.float32), tf.TensorSpec((th, tw, 2), tf.float32))
trds = tf.data.Dataset.from_generator(make_gen(Xtr, Ytr, a.seed), output_signature=sig).shuffle(512, seed=a.seed).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
vads = tf.data.Dataset.from_generator(make_gen(Xva, Yva, a.seed + 1), output_signature=sig).batch(batch).repeat().prefetch(tf.data.AUTOTUNE)
steps_tr = max(1, (Xtr.shape[0] * ppd) // batch)
steps_va = max(1, (Xva.shape[0] * ppd) // batch)
print(f"[{a.strategy} seed {a.seed}] {S['sampling']} ppd {ppd} steps {steps_tr}/{steps_va}", flush=True)

model = mtg.UNet((None, None, NC)); model.compile(optimizer="adam", loss=masked_mse)
es = EarlyStopping(patience=10, restore_best_weights=True)
model.fit(trds, epochs=50, steps_per_epoch=steps_tr, validation_data=vads,
          validation_steps=steps_va, callbacks=[es], verbose=2)
out = f'{c["MODEL_DIR"]}/{a.strategy}_seed{a.seed}.keras'
model.save(out); print("SAVED", out)

In [ ]:
# 5 strategies x 5 seeds = 25 runs, one process each (fresh GPU). Resumable (skip-existing).
import sys, subprocess
cfg = dict(SUBSET=SUBSET, train_year=train_year, train_range=train_range, val_range=val_range,
           test_range=test_range, features=features, DAY_BATCH=DAY_BATCH,
           ORIGINAL=ORIGINAL, RECHUNKED=RECHUNKED, MODEL_DIR=MODEL_DIR,
           SHARED_STATS=SHARED_STATS, STRATS=STRATS)
pickle.dump(cfg, open(f"{MODEL_DIR}/strat_config.pkl", "wb"))

for name in STRATS:
    for s in SEEDS:
        out = f"{MODEL_DIR}/{name}_seed{s}.keras"
        if os.path.exists(out):
            print("skip (exists):", out); continue
        print(f"\n===== {name} seed {s} =====", flush=True)
        rc = subprocess.run([sys.executable, "train_seed.py",
                             "--strategy", name, "--seed", str(s)]).returncode
        print(f"----- {name} seed {s} exited {rc} -----")
print("\nall done")

## Seed-averaged results

Run the two cells below once the sweep above has finished (they auto-skip any seed that is missing).

In [ ]:
# seed-averaged fake-cloud MAE / RMSE per strategy (mean +/- std over seeds)
def fc_scores(model, ds_std, dates):
    xv = [v for v in ds_std.data_vars if v != "CHL"]; ae = se = n = 0.0
    for d in dates:
        sub = ds_std.sel(time=d)
        X = np.stack([np.nan_to_num(sub[v].values, nan=0.0) for v in xv], -1).astype(np.float32)
        pred = model.predict(X[np.newaxis, ...], verbose=0)[0, :, :, 0] * y_std + y_mean
        truth = sub["CHL"].values * y_std + y_mean
        m = (sub["fake_cloud_flag"].values == 1) & np.isfinite(truth) & np.isfinite(pred)
        ae += np.abs(pred[m]-truth[m]).sum(); se += ((pred[m]-truth[m])**2).sum(); n += m.sum()
    return ae/n, np.sqrt(se/n)

print(f"{'strategy':>14} {'MAE mean':>9} {'MAE std':>8} {'RMSE mean':>10}  (n={len(SEEDS)} seeds)")
for name in STRATS:
    maes, rmses = [], []
    for s in SEEDS:
        p = f"{MODEL_DIR}/{name}_seed{s}.keras"
        if os.path.exists(p):
            mae, rmse = fc_scores(tf.keras.models.load_model(p, compile=False), ds_std_test, DATES)
            maes.append(mae); rmses.append(rmse)
    if maes:
        print(f"{name:>14} {np.mean(maes):9.4f} {np.std(maes):8.4f} {np.mean(rmses):10.4f}")

In [ ]:
# seed-averaged error over time: one line per strategy (mean of seeds), band = seed range
def compare_perf_seeds(strats, ds_std, seeds, freq="M", block=100):
    xv = [v for v in ds_std.data_vars if v != "CHL"]
    times = pd.to_datetime(ds_std.time.values)
    mdls, daily = {}, {}
    for name in strats:
        for s in seeds:
            p = f"{MODEL_DIR}/{name}_seed{s}.keras"
            if os.path.exists(p):
                mdls[(name, s)] = tf.keras.models.load_model(p, compile=False)
                daily[(name, s)] = np.full(len(times), np.nan)
    for i in range(0, len(times), block):
        sub = ds_std.isel(time=slice(i, i + block)).load()
        X = np.stack([np.nan_to_num(sub[v].values, nan=0.0) for v in xv], -1).astype(np.float32)
        truth = sub["CHL"].values * y_std + y_mean
        fake = (sub["fake_cloud_flag"].values == 1) & np.isfinite(truth)
        for key, mdl in mdls.items():
            pred = mdl.predict(X, batch_size=4, verbose=0)[..., 0] * y_std + y_mean
            m = fake & np.isfinite(pred)
            d = np.where(m, np.abs(pred - truth), np.nan)
            with np.errstate(invalid="ignore"):
                daily[key][i:i + d.shape[0]] = np.nanmean(d.reshape(d.shape[0], -1), axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    for name in strats:
        pers = []
        for s in seeds:
            if (name, s) not in daily: continue
            ser = pd.Series(daily[(name, s)], index=times).dropna()
            if freq in ("M", "Y"):
                ser = ser.groupby(ser.index.to_period(freq)).mean(); ser.index = ser.index.to_timestamp()
            pers.append(ser)
        if not pers: continue
        df = pd.concat(pers, axis=1); mean = df.mean(axis=1)
        ax.plot(mean.index, mean.values, lw=1.8, marker="o", ms=3, label=name)
        ax.fill_between(mean.index, df.min(axis=1), df.max(axis=1), alpha=0.12)
    ax.axvspan(pd.Timestamp(f"{train_year}-01-01"), pd.Timestamp(TRAIN_END), color="gray", alpha=0.08)
    ax.axvline(pd.Timestamp(VAL_END), ls="--", c="gray", lw=0.8)
    ax.set_xlabel("time"); ax.set_ylabel("fake-cloud MAE (log Chl-a)")
    ax.set_title(f"per-{freq} gap-fill error by strategy, mean of {len(seeds)} seeds (band = seed range)")
    ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3); plt.show()

compare_perf_seeds(list(STRATS), ds_std_full, SEEDS, freq="M")

## Coastal vs open-ocean error

Splits the seed-averaged fake-cloud MAE by proximity to shore, so the `coast` strategy is judged where it
is meant to help rather than only on the domain average. A pixel is "coastal" if within `COAST_PX` pixels
of land. If coast-anchoring works, `coast` should have the lowest **coastal** MAE.

In [ ]:
from scipy import ndimage
COAST_PX = 5    # a pixel is "coastal" if within this many pixels of land

land = np.isnan(ds_std_test["CHL"].values).all(0); ocean = ~land
coast_m = ocean & ndimage.binary_dilation(land, iterations=COAST_PX)
open_m  = ocean & ~coast_m
print(f"coastal pixels {int(coast_m.sum())} | open pixels {int(open_m.sum())} (COAST_PX={COAST_PX})\n")

def fc_split(model, ds_std, dates):
    xv = [v for v in ds_std.data_vars if v != "CHL"]; aec=nc=aeo=no=0.0
    for d in dates:
        sub = ds_std.sel(time=d)
        X = np.stack([np.nan_to_num(sub[v].values, nan=0.0) for v in xv], -1).astype(np.float32)
        pred = model.predict(X[np.newaxis, ...], verbose=0)[0, :, :, 0] * y_std + y_mean
        truth = sub["CHL"].values * y_std + y_mean
        fake = (sub["fake_cloud_flag"].values == 1) & np.isfinite(truth) & np.isfinite(pred)
        mc = fake & coast_m; mo = fake & open_m
        aec += np.abs(pred[mc]-truth[mc]).sum(); nc += mc.sum()
        aeo += np.abs(pred[mo]-truth[mo]).sum(); no += mo.sum()
    return aec/max(nc,1), aeo/max(no,1)

print(f"{'strategy':>14} {'coastal MAE':>12} {'open MAE':>10} {'coast-open':>11}")
for name in STRATS:
    cs, ops = [], []
    for s in SEEDS:
        p = f"{MODEL_DIR}/{name}_seed{s}.keras"
        if os.path.exists(p):
            m = tf.keras.models.load_model(p, compile=False)
            c, o = fc_split(m, ds_std_test, DATES); cs.append(c); ops.append(o)
    if cs:
        print(f"{name:>14} {np.mean(cs):12.4f} {np.mean(ops):10.4f} {np.mean(cs)-np.mean(ops):+11.4f}")